In [14]:
# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026 v8)
# Fuente dual: CSV metabólico (cluster) + CSV clínico
# ============================================================

# ===============================
# 1. CARGA DE LIBRERÍAS
# ===============================
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales", "effsize", "ggtext", "tibble", "stringr")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN
# ===============================
PATH_DATA_CLINICAL  <- "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_PCA_seleccion_reducida_conmerge-NUEVOSDATOS/DATA_MASTER_CLUSTERS_Y_CLINICA.csv"
PATH_DATA_METABOLIC <- "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_TumorPhenotype_UMAP_metrics_actualizado_sinl2/Merged_TumorPhenotype_UMAP_AllData_withClusters.csv"

# ── Columna de cluster a analizar ────────────────────────────────────────────
CLUSTER_COL <- "Cluster_UMAP_N50_D05_C2_Meuclidean_S100_DBSCAN_K2_S0.98_DB0.02_CH423206_Seed42"

# Etiquetas para los dos clusters
CLUSTER_LABELS <- c("0" = "Cluster 0", "1" = "Cluster 1")

# Colores para los grupos
group_colors <- c("Cluster 0" = "#0072B2", "Cluster 1" = "#D55E00")
COL_C0 <- "#0072B2"
COL_C1 <- "#D55E00"

# ─── TEMA GLOBAL ─────────────────────────────────────────────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 23, face = "bold"),
    axis.title    = element_text(size = 20),
    axis.text     = element_text(size = 18),
    legend.text   = element_text(size = 18),
    legend.title  = element_text(size = 19, face = "bold"),
    strip.text    = element_text(size = 19, face = "bold"),
    plot.subtitle = element_text(size = 18, color = "grey50")
  )

# ===============================
# 3. VARIABLES METABÓLICAS
# ===============================
SOLS <- c("FBA", "pFBA", "L1w")

expand_sols <- function(roots, sols = SOLS) {
  as.vector(outer(roots, sols, paste, sep = "_"))
}

SCALAR_ROOTS <- c(
  "CU", "EA", "WarburgIndex",
  "ATPConsumption", "ATPProduction",
  "RedoxIndex", "MFI", "AnabolismScore",
  "NADPHdemand", "TCA_completeness",
  "LipidSat", "LipidUnsat", "LipidPL",
  "GlnDependence"
)
SCALAR_VARS <- expand_sols(SCALAR_ROOTS)

ONCOMET_NAMES <- c("Lactate", "Succinate", "AlphaKG")
ONCOMET_VARS  <- as.vector(outer(
  paste0("Oncomet_", ONCOMET_NAMES), SOLS, paste, sep = "_"
))

ALL_METAB_VARS_STATIC <- c(SCALAR_VARS, ONCOMET_VARS)

# ===============================
# 4. CARGA Y PREPROCESAMIENTO
# ===============================

# ── Cargar datos metabólicos (contiene columna de cluster) ───────────────────
df_metabolic <- read.csv(PATH_DATA_METABOLIC, stringsAsFactors = FALSE)

if ("Model" %in% colnames(df_metabolic) && !"ModelName" %in% colnames(df_metabolic)) {
  df_metabolic <- df_metabolic %>% rename(ModelName = Model)
}
df_metabolic$ModelName <- toupper(gsub("\\.", "-", trimws(as.character(df_metabolic$ModelName))))
cat("🔬 Datos metabólicos cargados:", nrow(df_metabolic), "filas,", ncol(df_metabolic), "columnas\n")

# ── Cargar datos clínicos ────────────────────────────────────────────────────
df_clinical <- read.csv(PATH_DATA_CLINICAL, stringsAsFactors = FALSE)

if ("Model" %in% colnames(df_clinical) && !"ModelName" %in% colnames(df_clinical)) {
  df_clinical <- df_clinical %>% rename(ModelName = Model)
}
df_clinical$ModelName <- toupper(gsub("\\.", "-", trimws(as.character(df_clinical$ModelName))))
cat("📋 Datos clínicos cargados:", nrow(df_clinical), "filas,", ncol(df_clinical), "columnas\n")

# ── Verificar overlap ────────────────────────────────────────────────────────
common_models <- intersect(df_metabolic$ModelName, df_clinical$ModelName)
cat("\n🔗 Modelos en común:", length(common_models), "\n")
cat("   Solo en metabólico:", length(setdiff(df_metabolic$ModelName, df_clinical$ModelName)), "\n")
cat("   Solo en clínico:",    length(setdiff(df_clinical$ModelName,  df_metabolic$ModelName)), "\n")

# ── Merge: metabólico (left) + clínica ───────────────────────────────────────
df_raw <- df_metabolic %>%
  left_join(df_clinical, by = "ModelName", suffix = c("", ".clin"))
cat("\n✅ Dataset merged:", nrow(df_raw), "filas,", ncol(df_raw), "columnas\n")

# ── Verificar columna de cluster ─────────────────────────────────────────────
if (!CLUSTER_COL %in% colnames(df_raw)) {
  stop(paste0("❌ Columna de cluster no encontrada: '", CLUSTER_COL, "'\n",
              "   Columnas disponibles con 'Cluster':\n   ",
              paste(grep("Cluster|cluster", colnames(df_raw), value = TRUE), collapse = "\n   ")))
}

cat("\n🔍 Valores únicos en columna de cluster (raw):\n")
print(table(df_raw[[CLUSTER_COL]], useNA = "always"))

# ── Crear columna Group — excluir ruido DBSCAN (-1) ──────────────────────────
analysis <- df_raw %>%
  mutate(
    cluster_raw = as.character(!!sym(CLUSTER_COL)),
    Group = case_when(
      is.na(!!sym(CLUSTER_COL))              ~ NA_character_,
      cluster_raw == "-1"                    ~ NA_character_,   # ruido DBSCAN → excluir
      cluster_raw %in% names(CLUSTER_LABELS) ~ CLUSTER_LABELS[cluster_raw],
      TRUE                                   ~ paste0("Cluster ", cluster_raw)
    )
  ) %>%
  filter(!is.na(Group))

cat("\n✅ Pacientes por cluster (ruido DBSCAN -1 excluido):\n")
print(table(analysis$Group))

# ── Eliminar columnas duplicadas .clin ───────────────────────────────────────
dup_clin_cols <- grep("\\.clin$", colnames(analysis), value = TRUE)
cat("\n🔍 Columnas duplicadas (.clin) eliminadas:", length(dup_clin_cols), "\n")
if (length(dup_clin_cols) > 0) analysis <- analysis %>% select(-all_of(dup_clin_cols))

# ── Renombrar columnas con espacios ──────────────────────────────────────────
analysis <- analysis %>%
  rename_with(~ gsub("Menopausal\\.Status", "Menopausal Status", .x)) %>%
  rename_with(~ gsub("Cancer\\.Type",       "Cancer Type",       .x)) %>%
  rename_with(~ gsub("Genetic\\.Ancestry",  "Genetic Ancestry",  .x))

# ── Renombrar race / sex solo si no están ya con el nombre correcto ───────────
race_col <- grep("^race", colnames(analysis), ignore.case = TRUE, value = TRUE)
race_col <- race_col[race_col != "Race"][1]
sex_col  <- grep("^sex$|^gender$", colnames(analysis), ignore.case = TRUE, value = TRUE)
sex_col  <- sex_col[sex_col != "Sex"][1]

if (!is.na(race_col) && !is.null(race_col)) analysis <- analysis %>% rename(Race = !!sym(race_col))
if (!is.na(sex_col)  && !is.null(sex_col))  analysis <- analysis %>% rename(Sex  = !!sym(sex_col))

# ── Variables metabólicas disponibles ────────────────────────────────────────
sa_cols_csv <- grep(
  paste0("^SA_.+_(", paste(SOLS, collapse = "|"), ")$"),
  colnames(analysis), value = TRUE
)
cat("\n🔍 Subsistemas SA detectados:", length(sa_cols_csv), "\n")
if (length(sa_cols_csv) > 0) {
  subsys_unique <- unique(sub(paste0("_(", paste(SOLS, collapse = "|"), ")$"), "", sa_cols_csv))
  cat("   Subsistemas únicos:", length(subsys_unique), "\n")
  cat(paste0("   • ", head(subsys_unique, 10), collapse = "\n"), "\n")
  if (length(subsys_unique) > 10) cat("   ...", length(subsys_unique) - 10, "más\n")
}

ALL_METAB_VARS  <- c(ALL_METAB_VARS_STATIC, sa_cols_csv)
metab_available <- intersect(ALL_METAB_VARS, colnames(analysis))
metab_missing   <- setdiff(ALL_METAB_VARS, colnames(analysis))

cat("\n📊 Variables metabólicas encontradas:", length(metab_available), "/", length(ALL_METAB_VARS), "\n")
if (length(metab_missing) > 0) {
  cat("⚠️  NO encontradas (", length(metab_missing), "):\n")
  cat(paste0("   • ", head(metab_missing, 20), collapse = "\n"), "\n")
  if (length(metab_missing) > 20) cat("   ...", length(metab_missing) - 20, "más\n")
}

# ── Verificación variables clave ─────────────────────────────────────────────
key_vars <- c("OS.time", "OS", "Subtype", "ER", "PR", "HER2",
              "Genetic Ancestry", "Sex", "Race", "Menopausal Status",
              "ER_PR_HER2_Combo")
cat("\n📋 Verificación de variables clave:\n")
for (v in key_vars) {
  if (v %in% colnames(analysis)) {
    n_ok <- sum(!is.na(analysis[[v]]))
    cat(sprintf("   ✅ %-30s → %d/%d (%.1f%%)\n",
                v, n_ok, nrow(analysis), n_ok / nrow(analysis) * 100))
  } else {
    cat(sprintf("   ❌ %-30s → NO encontrada\n", v))
  }
}

# Nombres de los dos grupos
GROUP_NAMES <- sort(unique(analysis$Group))
cat("\n✅ Grupos activos:", paste(GROUP_NAMES, collapse = " | "), "\n")
for (g in GROUP_NAMES)
  cat(sprintf("      %s: %d pacientes\n", g, sum(analysis$Group == g)))

# ===============================
# 5A. GRUPOS CLÍNICOS
# ===============================
CLIN_GROUP_A <- intersect(c(
  "ER", "PR", "HER2", "Subtype", "ER_PR_HER2_Combo",
  "tumor_grade.diagnoses", "morphology.diagnoses", "primary_diagnosis.diagnoses"
), colnames(analysis))

CLIN_GROUP_B <- intersect(c(
  "ajcc_pathologic_stage.diagnoses", "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses", "ajcc_pathologic_m.diagnoses", "Metastasis_Flag"
), colnames(analysis))

CLIN_GROUP_C <- intersect(c(
  "prior_treatment.diagnoses", "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses", "Prior_Treatment_Flag"
), colnames(analysis))

CLIN_GROUP_D <- intersect(c(
  "age_at_diagnosis.diagnoses", "Menopausal Status", "Genetic Ancestry",
  "Race", "Sex", "Cancer Type",
  "OS.time", "OS"
), colnames(analysis))

CLIN_GROUPS <- Filter(function(v) length(v) > 0, list(
  "🧬 A. Biología Tumoral"        = CLIN_GROUP_A,
  "📊 B. Estadio y Carga Tumoral" = CLIN_GROUP_B,
  "💊 C. Historia Terapéutica"    = CLIN_GROUP_C,
  "🧍 D. Factores del Paciente"   = CLIN_GROUP_D
))

cat("\n📂 Grupos clínicos activos:", length(CLIN_GROUPS), "\n")
for (g in names(CLIN_GROUPS))
  cat("   •", g, ":", length(CLIN_GROUPS[[g]]), "variables\n")

# ===============================
# 5B. GRUPOS METABÓLICOS
# ===============================
make_group <- function(roots, available, sols = SOLS)
  intersect(expand_sols(roots, sols), available)

METAB_GROUPS <- Filter(function(v) length(v) > 0, list(
  "🔥 1. Energía y Bioenergética" = make_group(
    c("ATPConsumption", "ATPProduction", "WarburgIndex", "MFI", "CU", "EA"), metab_available),
  "🔴 2. Estado Redox y Anabolismo" = make_group(
    c("RedoxIndex", "NADPHdemand", "AnabolismScore", "TCA_completeness"), metab_available),
  "🧈 3. Metabolismo Lipídico" = make_group(
    c("LipidSat", "LipidUnsat", "LipidPL"), metab_available),
  "🧪 4. Dependencias Metabólicas Específicas" = c(
    make_group("GlnDependence", metab_available),
    intersect(ONCOMET_VARS, metab_available)),
  "🧩 5. Reprogramación Sistémica (SA_*)" = sa_cols_csv[sa_cols_csv %in% metab_available]
))

cat("\n📂 Grupos metabólicos activos:", length(METAB_GROUPS), "\n")
for (g in names(METAB_GROUPS))
  cat("   •", g, ":", length(METAB_GROUPS[[g]]), "variables\n")

# ===============================
# 5C. ANÁLISIS TNBC
# ===============================
tnbc_table   <- NULL
p_tnbc       <- NULL
p_tnbc_pct   <- NULL
p_tnbc_focus <- NULL
chi_label    <- ""

if ("ER_PR_HER2_Combo" %in% colnames(analysis)) {

  cat("\n🔍 Valores únicos en ER_PR_HER2_Combo:\n")
  print(table(analysis$ER_PR_HER2_Combo, useNA = "always"))

  analysis <- analysis %>%
    mutate(
      TNBC_Status = case_when(
        is.na(ER_PR_HER2_Combo)               ~ "Missing",
        ER_PR_HER2_Combo == "Triple_Negative" ~ "Triple Negativo",
        ER_PR_HER2_Combo == "Luminal_HR+"     ~ "Luminal HR+",
        ER_PR_HER2_Combo == "Luminal_HER2+"   ~ "Luminal HER2+",
        ER_PR_HER2_Combo == "HER2_Enriched"   ~ "HER2 Enriched",
        TRUE                                  ~ "Otro"
      ),
      TNBC_Status = factor(TNBC_Status, levels = c(
        "Triple Negativo", "Luminal HR+", "Luminal HER2+",
        "HER2 Enriched", "Otro", "Missing"
      ))
    )

  tnbc_table <- analysis %>%
    group_by(Group, TNBC_Status) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(Group, TNBC_Status, fill = list(n = 0)) %>%
    group_by(Group) %>%
    mutate(total = sum(n),
           pct   = round(n / total * 100, 1),
           label = paste0(n, "\n(", pct, "%)")) %>%
    ungroup()

  cat("\n📊 DISTRIBUCIÓN SUBTIPOS:\n")
  print(tnbc_table %>% select(Group, TNBC_Status, n, pct), n = 30)

  tbl_chi <- tnbc_table %>%
    filter(TNBC_Status != "Missing", n > 0) %>%
    select(Group, TNBC_Status, n) %>%
    pivot_wider(names_from = TNBC_Status, values_from = n, values_fill = 0) %>%
    column_to_rownames("Group") %>%
    as.matrix()

  p_chi_tnbc <- tryCatch(chisq.test(tbl_chi)$p.value, error = function(e) NA)
  chi_label  <- if (!is.na(p_chi_tnbc)) paste0("Chi² p = ", signif(p_chi_tnbc, 3)) else ""
  cat("\n📐 Chi² Subtipos vs Cluster:", chi_label, "\n")

  fill_vals_tnbc <- c(
    "Triple Negativo" = "#8B0000",
    "Luminal HR+"     = "#0072B2",
    "Luminal HER2+"   = "#E69F00",
    "HER2 Enriched"   = "#009E73",
    "Otro"            = "#CC79A7",
    "Missing"         = "grey75"
  )

  subtipo_order <- tnbc_table %>%
    filter(Group == GROUP_NAMES[1], n > 0) %>%
    arrange(desc(n)) %>%
    pull(TNBC_Status) %>% as.character()

  tnbc_plot_data <- tnbc_table %>%
    filter(n > 0) %>%
    mutate(TNBC_Status = factor(TNBC_Status, levels = subtipo_order))

  # Plot A: barras por subtipo, facet por cluster
  p_tnbc <- ggplot(tnbc_plot_data,
                   aes(x = TNBC_Status, y = n, fill = TNBC_Status)) +
    geom_col(color = "white", linewidth = 0.5, width = 0.65) +
    geom_label(aes(label = paste0(n, "\n(", pct, "%)"), y = n / 2),
               size = 4, fontface = "bold", color = "white",
               fill = NA, linewidth = 0, show.legend = FALSE) +
    scale_fill_manual(values = fill_vals_tnbc, drop = TRUE) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
    facet_wrap(~ Group, scales = "free_y", nrow = 1) +
    labs(title    = "Subtipos Moleculares: Conteo Absoluto por Cluster",
         subtitle = chi_label,
         x = NULL, y = "Número de Pacientes") +
    base_theme +
    theme(plot.title       = element_text(size = 16, face = "bold", hjust = 0.5),
          plot.subtitle    = element_text(size = 12, color = "grey40", hjust = 0.5),
          legend.position  = "none",
          strip.text       = element_text(size = 15, face = "bold", color = "white"),
          strip.background = element_rect(fill = "grey30", color = NA),
          axis.text.x      = element_text(size = 10, angle = 25, hjust = 1),
          axis.text.y      = element_text(size = 12),
          panel.spacing    = unit(1.5, "cm"))

  # ── Plot B: Proporciones apiladas 100% ──────────────────────────────────────
  p_tnbc_pct <- ggplot(tnbc_table %>% filter(n > 0) %>%
                         mutate(TNBC_Status = factor(TNBC_Status, levels = rev(subtipo_order))),
                       aes(x = Group, y = pct, fill = TNBC_Status)) +
    geom_col(position = "stack", color = "white", linewidth = 0.6, width = 0.5) +
    geom_text(
      aes(label = ifelse(pct >= 4, paste0(pct, "%\nn=", n), paste0(pct, "%")),
          size  = ifelse(pct >= 8, 6.0, 4.5)),
      position  = position_stack(vjust = 0.5),
      fontface  = "bold", color = "white", show.legend = FALSE
    ) +
    scale_size_identity() +
    scale_fill_manual(values = fill_vals_tnbc, name = "Subtipo", drop = TRUE) +
    scale_y_continuous(labels = function(x) paste0(x, "%"),
                       expand = expansion(mult = c(0, 0.02)),
                       limits = c(0, 102)) +
    labs(title    = "Subtipos Moleculares: Proporción (%)",
         subtitle = chi_label,
         x = "Cluster", y = "Porcentaje (%)") +
    base_theme +
    theme(plot.title      = element_text(size = 16, face = "bold", hjust = 0.5),
          plot.subtitle   = element_text(size = 12, color = "grey40", hjust = 0.5),
          legend.position = "right",
          legend.text     = element_text(size = 13),
          axis.text.x     = element_text(size = 15, face = "bold"))

  # ── Plot C: Triple Negativo vs Resto ────────────────────────────────────────
  tnbc_only <- tnbc_table %>%
    mutate(Es_TNBC = ifelse(TNBC_Status == "Triple Negativo",
                            "Triple Negativo", "Resto (incl. Missing)")) %>%
    group_by(Group, Es_TNBC) %>%
    summarise(n = sum(n), .groups = "drop") %>%
    group_by(Group) %>%
    mutate(total = sum(n),
           pct   = round(n / total * 100, 1),
           label = paste0(n, "\n(", pct, "%)")) %>%
    ungroup() %>%
    mutate(Es_TNBC = factor(Es_TNBC, levels = c("Triple Negativo", "Resto (incl. Missing)")))

  p_tnbc_focus <- ggplot(tnbc_only, aes(x = Group, y = n, fill = Es_TNBC)) +
    geom_col(position = "stack", color = "white", linewidth = 0.5) +
    geom_text(aes(label = label),
              position = position_stack(vjust = 0.5),
              size = 8, fontface = "bold", color = "white") +
    scale_fill_manual(values = c("Triple Negativo"       = "#8B0000",
                                 "Resto (incl. Missing)" = "grey65"),
                      name = NULL) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.08))) +
    labs(title    = "Triple Negativo vs Resto — por Cluster",
         subtitle = chi_label,
         x = "Cluster", y = "Número de Pacientes") +
    base_theme +
    theme(plot.title      = element_text(size = 20, face = "bold", hjust = 0.5),
          plot.subtitle   = element_text(size = 14, color = "grey40", hjust = 0.5),
          legend.position = "right")

} else {
  cat("\n⚠️  Columna 'ER_PR_HER2_Combo' no encontrada — análisis TNBC omitido.\n")
}

# ===============================
# 6. ANÁLISIS ESTADÍSTICO (FDR) + CLIFF'S DELTA
# ===============================

calc_cliffs_delta <- function(x_c1, x_c0) {
  nx <- length(x_c1); ny <- length(x_c0)
  if (nx == 0 || ny == 0) return(NA_real_)
  sum(outer(x_c1, x_c0, FUN = function(a, b) sign(a - b))) / (nx * ny)
}

label_effect <- function(d) {
  case_when(
    abs(d) >= 0.474 ~ "Grande",
    abs(d) >= 0.330 ~ "Mediano",
    abs(d) >= 0.147 ~ "Pequeño",
    TRUE            ~ "Negligible"
  )
}

fmt_stars <- function(p) {
  case_when(
    p < 0.001 ~ "***",
    p < 0.01  ~ "**",
    p < 0.05  ~ "*",
    TRUE      ~ "ns"
  )
}

G0 <- GROUP_NAMES[1]   # Cluster 0 (referencia)
G1 <- GROUP_NAMES[2]   # Cluster 1

run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))
    group_sizes <- table(sub_data$Group)

    if (length(group_sizes) != 2 || any(group_sizes < 3)) return(NULL)

    test     <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)
    meds     <- sub_data %>% group_by(Group) %>%
      summarise(m = median(value), .groups = "drop")
    med_g0   <- meds$m[meds$Group == G0]
    med_g1   <- meds$m[meds$Group == G1]
    x_g1     <- sub_data$value[sub_data$Group == G1]
    x_g0     <- sub_data$value[sub_data$Group == G0]
    delta    <- calc_cliffs_delta(x_g1, x_g0)
    pct_diff <- ifelse(med_g0 == 0, NA_real_,
                       (med_g1 - med_g0) / abs(med_g0) * 100)

    data.frame(
      Feature      = col,
      Med_G0       = med_g0,
      Med_G1       = med_g1,
      p_val        = test$p.value,
      cliffs_delta = delta,
      Pct_Diff     = pct_diff
    )
  })
  if (nrow(results) == 0) return(results)
  results %>%
    mutate(
      p_adj       = p.adjust(p_val, method = "fdr"),
      Fold_Change = Med_G1 / ifelse(Med_G0 == 0, NA, Med_G0),
      sig_stars   = fmt_stars(p_adj),
      effect_size = label_effect(cliffs_delta)
    ) %>%
    rename(!!paste0("Med_", G0) := Med_G0,
           !!paste0("Med_", G1) := Med_G1) %>%
    arrange(p_adj)
}

metab_res <- run_stats(analysis, metab_available)
cat("\n📈 Variables metabólicas significativas (p_adj < 0.05):",
    sum(metab_res$p_adj < 0.05, na.rm = TRUE), "/", nrow(metab_res), "\n")
write.csv(metab_res, "Estadisticas_Metabolicas_Clusters.csv", row.names = FALSE)

summary_by_group <- map_df(names(METAB_GROUPS), function(g) {
  vars_g <- METAB_GROUPS[[g]]
  res_g  <- metab_res %>% filter(Feature %in% vars_g)
  if (nrow(res_g) == 0) return(NULL)
  data.frame(
    Grupo         = g,
    n_vars        = length(vars_g),
    n_sig_p05     = sum(res_g$p_adj < 0.05, na.rm = TRUE),
    n_sig_p01     = sum(res_g$p_adj < 0.01, na.rm = TRUE),
    mejor_feature = res_g$Feature[which.min(res_g$p_adj)],
    mejor_p_adj   = min(res_g$p_adj, na.rm = TRUE),
    max_abs_delta = max(abs(res_g$cliffs_delta), na.rm = TRUE)
  )
})
write.csv(summary_by_group, "Resumen_Significancia_PorGrupo.csv", row.names = FALSE)
print(summary_by_group)

numeric_clinical_cols <- intersect(
  c("age_at_diagnosis.diagnoses", "is_ffpe.samples", "oct_embedded.samples",
    "Prior_Treatment_Flag", "Metastasis_Flag",
    "ER_PR_HER2_Combo_encoded", "Subtype_encoded",
    "OS.time", "OS"),
  colnames(analysis)
)
clinical_num_res <- run_stats(analysis, numeric_clinical_cols)

cat("\n📈 Variables clínicas numéricas analizadas:", length(numeric_clinical_cols), "\n")
if (nrow(clinical_num_res) > 0) {
  cat("   Significativas (p_adj < 0.05):",
      sum(clinical_num_res$p_adj < 0.05, na.rm = TRUE), "\n")
  print(clinical_num_res %>% select(Feature, cliffs_delta, p_adj, sig_stars, effect_size))
}

# ===============================
# 7. FUNCIONES DE PLOTEO
# ===============================

plot_box_feat <- function(feat, data, stats_df) {
  if (!feat %in% colnames(data)) return(NULL)
  p_info <- stats_df %>% filter(Feature == feat) %>% slice(1)
  if (nrow(p_info) == 0) return(NULL)

  sig_color    <- if (!is.na(p_info$p_adj) && p_info$p_adj < 0.05) "#e63946" else "grey60"
  subtitle_txt <- paste0(
    "p-adj: ", signif(p_info$p_adj, 3),
    "  |  δ=", round(p_info$cliffs_delta, 2),
    " (", p_info$effect_size, ")",
    if (!is.na(p_info$Pct_Diff)) paste0("  |  Δ%: ", round(p_info$Pct_Diff, 1)) else ""
  )

  plot_data          <- data %>% filter(!is.na(Group))
  vals               <- sort(unique(na.omit(plot_data[[feat]])))
  is_binary          <- is.numeric(plot_data[[feat]]) && length(vals) == 2 && all(vals %in% c(0, 1))
  is_survival_time   <- grepl("OS\\.time|survival.*time|time.*survival", feat, ignore.case = TRUE)
  is_survival_status <- feat == "OS" || grepl("survival.*status|status.*survival", feat, ignore.case = TRUE)
  is_sex             <- grepl("^sex$|^gender$", feat, ignore.case = TRUE)

  if (is_survival_status || is_binary) {
    plot_data <- plot_data %>%
      mutate(Category = case_when(
        is.na(!!sym(feat))                    ~ "Missing",
        is_survival_status & !!sym(feat) == 1 ~ "Fallecido (1)",
        is_survival_status & !!sym(feat) == 0 ~ "Vivo (0)",
        !!sym(feat) == 1                      ~ "Sí (1)",
        !!sym(feat) == 0                      ~ "No (0)",
        TRUE                                  ~ "Missing"
      )) %>%
      mutate(Category = factor(Category,
               levels = c(if (is_survival_status) c("Vivo (0)", "Fallecido (1)")
                          else c("No (0)", "Sí (1)"), "Missing")))

    tbl   <- table(plot_data$Category[plot_data$Category != "Missing"],
                   plot_data$Group[plot_data$Category != "Missing"])
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    chi_label_local <- if (!is.na(p_chi)) paste0("Chi² p=", signif(p_chi, 3)) else ""

    count_df <- plot_data %>%
      group_by(Group, Category) %>%
      summarise(n = n(), .groups = "drop") %>%
      group_by(Group) %>%
      mutate(pct   = round(n / sum(n) * 100, 1),
             label = paste0(n, "\n(", pct, "%)"))

    fill_vals_local <- if (is_survival_status)
      c("Vivo (0)" = COL_C0, "Fallecido (1)" = COL_C1, "Missing" = "grey80")
    else
      c("No (0)"   = COL_C0, "Sí (1)"        = COL_C1, "Missing" = "grey80")

    ggplot(count_df, aes(x = Group, y = pct, fill = Category)) +
      geom_col(position = "stack", color = "white", linewidth = 0.6, width = 0.5) +
      geom_text(
        aes(label = label,
            size  = ifelse(pct >= 8, 6.0, 4.5)),
        position  = position_stack(vjust = 0.5),
        fontface  = "bold", color = "white", show.legend = FALSE
      ) +
      scale_size_identity() +
      scale_fill_manual(values = fill_vals_local, name = NULL) +
      scale_y_continuous(labels = function(x) paste0(x, "%"),
                         expand = expansion(mult = c(0, 0.02)),
                         limits = c(0, 102)) +
      labs(title    = feat,
           subtitle = paste0(subtitle_txt, "  |  ", chi_label_local),
           x = "Cluster", y = "Porcentaje (%)") +
      base_theme +
      theme(legend.position = "right",
            legend.text     = element_text(size = 13),
            plot.title      = element_text(size = 17, color = sig_color, face = "bold",
                                           hjust = 0.5),
            plot.subtitle   = element_text(size = 12, color = "grey40", hjust = 0.5),
            axis.text.x     = element_text(size = 15, face = "bold"))

  } else if (is_sex) {
    plot_data <- plot_data %>%
      mutate(Category = case_when(
        is.na(!!sym(feat)) | trimws(tolower(!!sym(feat))) %in% c("", "na", "nan", "unknown") ~ "Missing",
        TRUE ~ str_to_title(trimws(as.character(!!sym(feat))))
      ))
    count_df <- plot_data %>%
      group_by(Group, Category) %>%
      summarise(n = n(), .groups = "drop") %>%
      group_by(Group) %>%
      mutate(pct   = round(n / sum(n) * 100, 1),
             label = paste0(n, "\n(", pct, "%)"))
    cats_sex <- setdiff(unique(count_df$Category), "Missing")
    pal_sex  <- c(setNames(scales::hue_pal()(length(cats_sex)), cats_sex),
                  "Missing" = "grey80")
    tbl   <- table(plot_data$Category[plot_data$Category != "Missing"],
                   plot_data$Group[plot_data$Category != "Missing"])
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    chi_label_local <- if (!is.na(p_chi)) paste0("Chi² p=", signif(p_chi, 3)) else ""

    ggplot(count_df, aes(x = Group, y = pct, fill = Category)) +
      geom_col(position = "stack", color = "white", linewidth = 0.6, width = 0.5) +
      geom_text(
        aes(label = label,
            size  = ifelse(pct >= 8, 6.0, 4.5)),
        position  = position_stack(vjust = 0.5),
        fontface  = "bold", color = "white", show.legend = FALSE
      ) +
      scale_size_identity() +
      scale_fill_manual(values = pal_sex, name = feat) +
      scale_y_continuous(labels = function(x) paste0(x, "%"),
                         expand = expansion(mult = c(0, 0.02)),
                         limits = c(0, 102)) +
      labs(title    = feat,
           subtitle = paste0(subtitle_txt, "  |  ", chi_label_local),
           x = "Cluster", y = "Porcentaje (%)") +
      base_theme +
      theme(legend.position = "right",
            legend.text     = element_text(size = 13),
            plot.title      = element_text(size = 17, color = sig_color, face = "bold",
                                           hjust = 0.5),
            plot.subtitle   = element_text(size = 12, color = "grey40", hjust = 0.5),
            axis.text.x     = element_text(size = 15, face = "bold"))

  } else if (is_survival_time) {
    plot_data <- plot_data %>%
      mutate(has_data = !is.na(!!sym(feat)) & is.finite(!!sym(feat)),
             value    = as.numeric(!!sym(feat)))
    n_miss <- plot_data %>%
      group_by(Group) %>%
      summarise(n_miss = sum(!has_data), n_total = n(), .groups = "drop") %>%
      mutate(pct   = round(n_miss / n_total * 100, 1),
             label = paste0(n_miss, "\n(", pct, "%)"))
    p1 <- ggplot(plot_data %>% filter(has_data),
                 aes(x = Group, y = value, fill = Group)) +
      geom_violin(alpha = 0.2, color = NA) +
      geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
      scale_fill_manual(values = group_colors) +
      labs(title = feat, subtitle = subtitle_txt, x = "Cluster", y = "Tiempo (días)") +
      base_theme +
      theme(legend.position = "none",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 12))
    p2 <- ggplot(n_miss, aes(x = Group, y = n_miss, fill = Group)) +
      geom_col(width = 0.5, alpha = 0.7) +
      geom_text(aes(label = label), vjust = -0.3, size = 4.5, fontface = "bold") +
      scale_fill_manual(values = group_colors) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
      labs(title = "Missing", x = "Cluster", y = "N pacientes") +
      base_theme +
      theme(legend.position = "none",
            plot.title = element_text(size = 14, color = "grey50"))
    return(gridExtra::arrangeGrob(p1, p2, ncol = 2, widths = c(3, 1)))

  } else {
    plot_data <- plot_data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
    ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
      geom_violin(alpha = 0.2, color = NA) +
      geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
      labs(title = feat, subtitle = subtitle_txt, y = NULL, x = "Cluster") +
      scale_fill_manual(values = group_colors) +
      base_theme +
      theme(legend.position = "none",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 13))
  }
}

plot_heatmap_cliffs <- function(stats_df, group_name = "Todas", max_vars = 40) {
  if (is.null(stats_df) || nrow(stats_df) == 0) return(NULL)
  hm_data <- stats_df %>%
    filter(!is.na(cliffs_delta)) %>%
    arrange(p_adj) %>% head(max_vars) %>%
    mutate(Feature   = factor(Feature, levels = Feature[order(cliffs_delta)]),
           sig_mark  = fmt_stars(p_adj),
           label_txt = paste0(sig_mark, " δ=", round(cliffs_delta, 2)))
  ggplot(hm_data, aes(x = 1, y = Feature, fill = cliffs_delta)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = label_txt,
                  color = ifelse(abs(cliffs_delta) > 0.30, "white", "grey20")),
              size = 3.8, fontface = "bold") +
    scale_fill_gradient2(low = COL_C0, mid = "white", high = COL_C1,
                         midpoint = 0, limits = c(-1, 1),
                         name = paste0("Cliff's δ\n", G1, " vs ", G0)) +
    scale_color_identity() +
    scale_x_continuous(breaks = NULL) +
    labs(title    = paste("Cliff's δ:", group_name),
         subtitle = paste0(G0, " < 0 < ", G1, "  |  * p_adj<0.05"),
         x = NULL, y = NULL) +
    base_theme +
    theme(axis.text.y   = element_text(size = 13),
          plot.subtitle = element_text(size = 14, color = "grey45"))
}

plot_forest_cliffs <- function(stats_df, title_txt = "Forest Plot — Cliff's δ", top_n = 40) {
  if (is.null(stats_df) || nrow(stats_df) == 0) return(NULL)
  df <- stats_df %>%
    filter(!is.na(cliffs_delta), !is.na(p_adj)) %>%
    arrange(p_adj) %>% head(top_n) %>%
    mutate(
      pt_color  = case_when(
        p_adj >= 0.05    ~ "grey65",
        cliffs_delta > 0 ~ COL_C1,
        TRUE             ~ COL_C0),
      ann_label = paste0(fmt_stars(p_adj),
                         "  δ=", round(cliffs_delta, 2),
                         "  (", round(Pct_Diff, 1), "%)"),
      Feature   = factor(Feature, levels = Feature[order(cliffs_delta)])
    )
  ggplot(df, aes(x = cliffs_delta, y = Feature)) +
    annotate("rect", xmin = -0.474, xmax =  0.474, ymin = -Inf, ymax = Inf, fill = "grey96") +
    annotate("rect", xmin = -0.330, xmax =  0.330, ymin = -Inf, ymax = Inf, fill = "grey92") +
    annotate("rect", xmin = -0.147, xmax =  0.147, ymin = -Inf, ymax = Inf, fill = "grey87") +
    geom_vline(xintercept = 0, linetype = "solid", color = "grey40", linewidth = 0.6) +
    geom_segment(aes(x = 0, xend = cliffs_delta, yend = Feature, color = pt_color),
                 linewidth = 0.7, alpha = 0.6) +
    geom_point(aes(color = pt_color, size = abs(cliffs_delta)), alpha = 0.95) +
    geom_text(aes(label = ann_label, color = pt_color,
                  hjust = ifelse(cliffs_delta >= 0, -0.12, 1.12)),
              size = 5.5, fontface = "bold") +
    scale_color_identity() +
    scale_size_continuous(range = c(2.5, 8), guide = "none") +
    scale_x_continuous(
      limits = c(-1.1, 1.1),
      breaks = c(-0.6, -0.474, -0.330, -0.147, 0, 0.147, 0.330, 0.474, 0.6),
      labels = c("-0.6", "Grande", "Mediano", "Pequeño",
                 "0", "Pequeño", "Mediano", "Grande", "0.6")
    ) +
    labs(title    = title_txt,
         subtitle = paste0("Cliff's δ + Δ% mediana | ",
                           "<span style='color:", COL_C0, "'>●</span> mayor en ", G0,
                           " · <span style='color:", COL_C1, "'>●</span> mayor en ", G1),
         x = "Cliff's delta", y = NULL) +
    theme_classic(base_size = 16) +
    theme(plot.title         = element_text(size = 18, face = "bold", hjust = 0.5),
          plot.subtitle      = element_markdown(size = 14, hjust = 0.5, color = "grey45"),
          axis.text.y        = element_text(size = 14),
          axis.text.x        = element_text(size = 11, angle = 35, hjust = 1),
          panel.grid.major.x = element_line(color = "grey88", linewidth = 0.4))
}

# ── plot_clinical: estilo p_tnbc_pct (barras apiladas % con etiquetas n y %) ─
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)
  tmp[[var]] <- trimws(as.character(tmp[[var]]))

  if (is.numeric(data[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 6) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, x = "Cluster") + base_theme

  } else {
    tbl   <- table(tmp[[var]], tmp$Group)
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    subtitle_txt <- if (!is.na(p_chi)) paste("Chi² p:", signif(p_chi, 3)) else ""

    count_df <- tmp %>%
      group_by(Group, !!sym(var)) %>%
      summarise(n = n(), .groups = "drop") %>%
      group_by(Group) %>%
      mutate(
        pct     = round(n / sum(n) * 100, 1),
        cum_top = cumsum(pct),
        y_mid   = cum_top - pct / 2    # centro exacto del segmento
      ) %>%
      ungroup() %>%
      mutate(grp_num = as.numeric(factor(Group, levels = unique(Group))))

    # Tres niveles:
    # >= 8%  → dentro, etiqueta completa pct% + n=
    # 3–8%   → dentro, solo pct%
    # < 3%   → fuera a la derecha, negro

    ggplot(count_df, aes(x = Group, y = pct, fill = !!sym(var))) +
      geom_col(position = "stack", color = "white", linewidth = 0.6, width = 0.5) +
      # Solo segmentos grandes (>= 8%) muestran etiqueta dentro
      geom_text(
        data      = ~ filter(.x, pct >= 8),
        aes(label = paste0(pct, "%\nn=", n)),
        position  = position_stack(vjust = 0.5),
        size = 6.0, fontface = "bold", color = "white", show.legend = FALSE
      ) +
      scale_y_continuous(labels = function(x) paste0(x, "%"),
                         expand = expansion(mult = c(0, 0.02)),
                         limits = c(0, 102)) +
      coord_cartesian(clip = "off") +
      labs(title    = var,
           subtitle = subtitle_txt,
           x = "Cluster", y = "Porcentaje (%)") +
      base_theme +
      theme(plot.title      = element_text(size = 17, face = "bold", hjust = 0.5),
            plot.subtitle   = element_text(size = 12, color = "grey40", hjust = 0.5),
            legend.position = "right",
            legend.text     = element_text(size = 13),
            axis.text.x     = element_text(size = 15, face = "bold"),
            plot.margin     = margin(5, 80, 5, 5))
  }
}

plot_metab_vs_clinical <- function(metab_feat, clin_var, data) {
  if (!metab_feat %in% colnames(data)) return(NULL)
  if (!clin_var   %in% colnames(data)) return(NULL)
  tmp <- data %>%
    filter(!is.na(!!sym(metab_feat)), !is.na(!!sym(clin_var)),
           is.finite(!!sym(metab_feat))) %>%
    mutate(ClinVar = trimws(as.character(!!sym(clin_var))))
  if (nrow(tmp) == 0 || length(unique(tmp$ClinVar)) < 2) return(NULL)
  ggplot(tmp, aes(x = ClinVar, y = !!sym(metab_feat), fill = Group)) +
    geom_boxplot(outlier.shape = 16, outlier.size = 1.5, alpha = 0.8) +
    scale_fill_manual(values = group_colors) +
    labs(title    = paste0(metab_feat, " × ", clin_var),
         subtitle = "¿La diferencia entre clusters es independiente del fenotipo clínico?",
         x = clin_var, y = metab_feat) +
    base_theme +
    theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 13))
}

# ===============================
# 8. GENERACIÓN DEL REPORTE PDF
# ===============================
algo_name <- gsub("Cluster_", "", CLUSTER_COL)
pdf_name  <- paste0("Reporte_", gsub("[^A-Za-z0-9]", "_", algo_name), ".pdf")
pdf(pdf_name, width = 14, height = 11)

# ─── PÁGINA DE TÍTULO ────────────────────────────────────────────────────────
grid.newpage()
grid.text(paste0("Reporte: ", G0, " vs ", G1),
          gp = gpar(fontsize = 28, fontface = "bold"), y = 0.78)
grid.text(paste0("Algoritmo: ", CLUSTER_COL),
          gp = gpar(fontsize = 13, col = "grey50"), y = 0.68)
grid.text(paste0(G0, ": ", sum(analysis$Group == G0), " pacientes",
                 "   |   ", G1, ": ", sum(analysis$Group == G1), " pacientes"),
          gp = gpar(fontsize = 21), y = 0.57)
grid.text(paste("Variables metabólicas analizadas:", length(metab_available),
                "  |  Significativas (p_adj<0.05):", sum(metab_res$p_adj < 0.05, na.rm = TRUE)),
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.47)
grid.text(paste("Soluciones FBA:", paste(SOLS, collapse = ", ")),
          gp = gpar(fontsize = 17, col = "grey50"), y = 0.37)
grid.text("Estructura: Clínica → TNBC → Metabolismo → Integración → Supervivencia",
          gp = gpar(fontsize = 17, col = "#0072B2"), y = 0.27)
grid.text("Tamaño de efecto: Cliff's δ  |  ≥0.147 pequeño · ≥0.330 mediano · ≥0.474 grande",
          gp = gpar(fontsize = 15, col = "grey55"), y = 0.17)

# ─── SECCIÓN I: CLÍNICA ──────────────────────────────────────────────────────
grid.newpage()
grid.text(paste0("SECCIÓN I\nDiferencias Clínicas: ", G0, " vs ", G1),
          gp = gpar(fontsize = 33, fontface = "bold", col = "#0072B2"), y = 0.60)
grid.text("Macro-categorías: Biología Tumoral | Estadio | Historia Terapéutica | Factores del Paciente",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

for (clin_gname in names(CLIN_GROUPS)) {
  vars_clin    <- CLIN_GROUPS[[clin_gname]]
  numeric_vars <- vars_clin[sapply(vars_clin, function(v) is.numeric(analysis[[v]]))]
  categ_vars   <- setdiff(vars_clin, numeric_vars)

  if (length(numeric_vars) > 0) {
    num_plots_clin <- map(numeric_vars, ~plot_box_feat(.x, analysis, clinical_num_res)) %>%
      Filter(Negate(is.null), .)
    if (length(num_plots_clin) > 0) {
      for (i in seq(1, length(num_plots_clin), by = 6)) {
        idx <- i:min(i + 5, length(num_plots_clin))
        grid.arrange(grobs = num_plots_clin[idx],
                     ncol = min(3, length(idx)),
                     nrow = ceiling(length(idx) / min(3, length(idx))),
                     top  = textGrob(paste("Clínica Numérica —", clin_gname),
                                     gp = gpar(fontsize = 21, fontface = "bold")))
      }
    }
    clin_hm <- plot_heatmap_cliffs(
      clinical_num_res %>% filter(Feature %in% numeric_vars),
      group_name = paste("Clínica:", clin_gname), max_vars = length(numeric_vars))
    if (!is.null(clin_hm)) print(clin_hm)
    clin_forest <- plot_forest_cliffs(
      clinical_num_res %>% filter(Feature %in% numeric_vars),
      title_txt = paste("Cliff's δ — Clínica:", clin_gname), top_n = length(numeric_vars))
    if (!is.null(clin_forest)) print(clin_forest)
  }

  if (length(categ_vars) > 0) {
    cat_plots_clin <- map(categ_vars, ~plot_clinical(.x, analysis)) %>%
      Filter(Negate(is.null), .)
    if (length(cat_plots_clin) > 0) {
      for (i in seq(1, length(cat_plots_clin), by = 2)) {
        idx <- i:min(i + 1, length(cat_plots_clin))
        grid.arrange(grobs = cat_plots_clin[idx], nrow = length(idx), ncol = 1,
                     top = textGrob(paste("Clínica Categórica —", clin_gname),
                                    gp = gpar(fontsize = 21, fontface = "bold")))
      }
    }
  }
}

# ─── SECCIÓN I-B: TNBC ───────────────────────────────────────────────────────
if (!is.null(tnbc_table)) {
  grid.newpage()
  grid.text(paste0("SECCIÓN I-B\nAnálisis de Subtipos Moleculares: ", G0, " vs ", G1),
            gp = gpar(fontsize = 30, fontface = "bold", col = "#8B0000"), y = 0.60)
  grid.text("Fuente: ER_PR_HER2_Combo  |  Triple Negativo · Luminal HR+ · Luminal HER2+ · HER2 Enriched · Missing",
            gp = gpar(fontsize = 16, col = "grey40"), y = 0.42)

  print(p_tnbc_pct)
  print(p_tnbc_focus)

  tnbc_summary <- tnbc_table %>%
    filter(TNBC_Status == "Triple Negativo") %>%
    select(Group, n_TNBC = n, total, pct_TNBC = pct)
  summary_text <- paste0(
    "Resumen Triple Negativo (fuente: ER_PR_HER2_Combo):\n\n",
    paste(apply(tnbc_summary, 1, function(r)
      sprintf("  %s:  %s TNBC de %s totales (%.1f%%)",
              r["Group"], r["n_TNBC"], r["total"], as.numeric(r["pct_TNBC"]))
    ), collapse = "\n"))
  grid.newpage()
  grid.text(summary_text, gp = gpar(fontsize = 22, col = "#8B0000"), y = 0.62, just = "centre")
  grid.text(paste0("Chi² test subtipos vs Cluster: ", chi_label),
            gp = gpar(fontsize = 18, col = "grey40"), y = 0.40)
}

# ─── SECCIÓN II: METABOLISMO ─────────────────────────────────────────────────
grid.newpage()
grid.text(paste0("SECCIÓN II\nDiferencias Metabólicas: ", G0, " vs ", G1),
          gp = gpar(fontsize = 33, fontface = "bold", col = "#D55E00"), y = 0.60)
grid.text("5 bloques funcionales: Energía | Redox | Lípidos | Dependencias | Reprogramación Sistémica",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

if (nrow(metab_res) > 0) {
  ff <- plot_forest_cliffs(metab_res, "Top 40 variables metabólicas — Cliff's δ", top_n = 40)
  if (!is.null(ff)) print(ff)
}

for (group_name in names(METAB_GROUPS)) {
  vars_in_group <- METAB_GROUPS[[group_name]]
  stats_group   <- metab_res %>% filter(Feature %in% vars_in_group)
  if (nrow(stats_group) == 0) next
  hm_g <- plot_heatmap_cliffs(stats_group, group_name, max_vars = 40)
  if (!is.null(hm_g)) print(hm_g)
  ff_g <- plot_forest_cliffs(stats_group,
                              title_txt = paste("Cliff's δ —", group_name),
                              top_n = min(40, nrow(stats_group)))
  if (!is.null(ff_g)) print(ff_g)
  plots_g <- map(vars_in_group, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(plots_g) > 0) {
    for (i in seq(1, length(plots_g), by = 6)) {
      idx <- i:min(i + 5, length(plots_g))
      grid.arrange(grobs = plots_g[idx],
                   ncol = min(3, length(idx)),
                   nrow = ceiling(length(idx) / min(3, length(idx))),
                   top  = textGrob(paste("Metabolismo —", group_name),
                                   gp = gpar(fontsize = 21, fontface = "bold")))
    }
  }
}

top12 <- metab_res %>%
  filter(!is.na(p_adj), !is.na(cliffs_delta)) %>%
  arrange(p_adj, desc(abs(cliffs_delta))) %>%
  head(12)

if (nrow(top12) >= 2) {
  hm_top12 <- plot_heatmap_cliffs(top12, "Top 12 Variables", max_vars = 12)
  if (!is.null(hm_top12)) print(hm_top12)
  ff_top12 <- plot_forest_cliffs(top12, title_txt = "🏆 Top 12 Variables — Cliff's δ", top_n = 12)
  if (!is.null(ff_top12)) print(ff_top12)
  top12_plots <- map(top12$Feature, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(top12_plots) > 0) {
    for (i in seq(1, length(top12_plots), by = 6)) {
      idx <- i:min(i + 5, length(top12_plots))
      grid.arrange(grobs = top12_plots[idx],
                   ncol = min(3, length(idx)),
                   nrow = ceiling(length(idx) / min(3, length(idx))),
                   top  = textGrob("🏆 Top 12 Variables Metabólicas",
                                   gp = gpar(fontsize = 23, fontface = "bold")))
    }
  }
}

# ─── SECCIÓN III: INTEGRACIÓN ────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN III\nAnálisis Integrado: Clínica ↔ Metabolismo",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#009E73"), y = 0.60)
grid.text("¿Los clusters reflejan el subtipo? ¿O reprogramación metabólica independiente?",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

integration_clin_vars <- intersect(
  c("Subtype", "ajcc_pathologic_stage.diagnoses", "ER", "HER2",
    "Metastasis_Flag", "ER_PR_HER2_Combo"),
  colnames(analysis))
top_features <- top12$Feature

for (clin_var in integration_clin_vars) {
  integ_plots <- map(head(top_features, 3),
                     ~plot_metab_vs_clinical(.x, clin_var, analysis)) %>%
    Filter(Negate(is.null), .)
  if (length(integ_plots) > 0) {
    grid.arrange(grobs = integ_plots, ncol = min(3, length(integ_plots)), nrow = 1,
                 top = textGrob(paste0("Integración: Top metabólicas × ", clin_var),
                                gp = gpar(fontsize = 21, fontface = "bold", col = "#009E73")))
  }
}

if ("ER_PR_HER2_Combo" %in% colnames(analysis) && length(top_features) > 0) {
  subtype_metab <- analysis %>%
    filter(!is.na(ER_PR_HER2_Combo)) %>%
    select(Group, Subtype = ER_PR_HER2_Combo,
           all_of(intersect(top_features, colnames(analysis)))) %>%
    pivot_longer(cols = -c(Group, Subtype), names_to = "Feature", values_to = "Value") %>%
    group_by(Group, Subtype, Feature) %>%
    summarise(Median = median(Value, na.rm = TRUE), .groups = "drop") %>%
    group_by(Feature) %>%
    mutate(Median_z = scale(Median)[, 1]) %>%
    ungroup() %>%
    mutate(GroupSubtype = paste0(Group, "\n", Subtype))
  if (nrow(subtype_metab) > 0) {
    print(ggplot(subtype_metab, aes(x = GroupSubtype, y = Feature, fill = Median_z)) +
            geom_tile(color = "white", linewidth = 0.6) +
            scale_fill_gradient2(low = COL_C0, mid = "white", high = COL_C1,
                                 midpoint = 0, name = "Mediana (z-score)") +
            labs(title    = "Firma Metabólica por Subtipo Molecular",
                 subtitle = paste0("Diferencias metabólicas entre ", G0, " y ", G1),
                 x = "Cluster · Subtipo", y = NULL) +
            base_theme +
            theme(plot.title        = element_text(size = 24, face = "bold", hjust = 0.5),
                  plot.subtitle     = element_text(size = 16, color = "grey40", hjust = 0.5),
                  axis.text.x       = element_text(angle = 40, hjust = 1, size = 13),
                  axis.text.y       = element_text(size = 14),
                  legend.key.width  = unit(1.5, "cm"),
                  legend.key.height = unit(0.5, "cm"),
                  legend.direction  = "horizontal",
                  legend.position   = "bottom"))
  }
}

if ("Subtype" %in% colnames(analysis) && length(top_features) > 0) {
  subtype_metab <- analysis %>%
    filter(!is.na(Subtype)) %>%
    select(Group, Subtype, all_of(intersect(top_features, colnames(analysis)))) %>%
    pivot_longer(cols = -c(Group, Subtype), names_to = "Feature", values_to = "Value") %>%
    group_by(Group, Subtype, Feature) %>%
    summarise(Median = median(Value, na.rm = TRUE), .groups = "drop") %>%
    group_by(Feature) %>%
    mutate(Median_z = scale(Median)[, 1]) %>%
    ungroup() %>%
    mutate(GroupSubtype = paste0(Group, "\n", Subtype))

  if (nrow(subtype_metab) > 0) {
    print(
      ggplot(subtype_metab, aes(x = GroupSubtype, y = Feature, fill = Median_z)) +
        geom_tile(color = "white", linewidth = 0.6) +
        scale_fill_gradient2(low = COL_C0, mid = "white", high = COL_C1,
                             midpoint = 0, name = "Mediana (z-score)") +
        labs(title    = "Firma Metabólica por Subtipo Molecular",
             subtitle = paste0("Diferencias metabólicas entre ", G0, " y ", G1),
             x        = "Cluster · Subtipo", y = NULL) +
        base_theme +
        theme(plot.title        = element_text(size = 24, face = "bold", hjust = 0.5),
              plot.subtitle     = element_text(size = 16, color = "grey40", hjust = 0.5),
              axis.text.x       = element_text(angle = 40, hjust = 1, size = 15),
              axis.text.y       = element_text(size = 14),
              axis.title.x      = element_text(size = 16, face = "bold"),
              legend.title      = element_text(size = 14, face = "bold"),
              legend.text       = element_text(size = 13),
              legend.key.width  = unit(1.5, "cm"),
              legend.key.height = unit(0.5, "cm"),
              legend.direction  = "horizontal",
              legend.position   = "bottom",
              panel.spacing     = unit(0.3, "lines"))
    )
  }
}

# ─── SECCIÓN IV: KAPLAN-MEIER ────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN IV\nAnálisis de Supervivencia",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#CC79A7"), y = 0.60)
grid.text(paste0("¿El cluster predice pronóstico? (", G0, " vs ", G1, ")"),
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>% filter(!is.na(OS.time), !is.na(OS), OS.time > 0)
  cat("\n📊 KM pacientes:", nrow(km_data),
      "(", G0, ":", sum(km_data$Group == G0),
      "| ", G1, ":", sum(km_data$Group == G1), ")\n")

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval = TRUE, pval.size = 6, conf.int = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE, risk.table.cex = 1.2,
                     surv.line.size = 1.2,
                     title          = paste0("Supervivencia: ", G0, " vs ", G1),
                     xlab           = "Tiempo (días)", ylab = "Probabilidad de Supervivencia",
                     legend.labs    = GROUP_NAMES,
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 27, face = "bold"),
                             axis.title  = element_text(size = 21),
                             axis.text   = element_text(size = 19),
                             legend.text = element_text(size = 19))))

    if ("ER_PR_HER2_Combo" %in% colnames(km_data)) {
      subtypes_km <- km_data %>%
        filter(!is.na(ER_PR_HER2_Combo)) %>%
        group_by(ER_PR_HER2_Combo) %>%
        filter(n_distinct(Group) == 2, n() >= 10) %>%
        pull(ER_PR_HER2_Combo) %>% unique()
      for (stype in subtypes_km) {
        km_sub <- km_data %>% filter(ER_PR_HER2_Combo == stype)
        if (nrow(km_sub) < 10) next
        fit_sub <- survfit(Surv(OS.time, OS) ~ Group, data = km_sub)
        tryCatch(
          print(ggsurvplot(fit_sub, data = km_sub,
                           pval = TRUE, pval.size = 5, conf.int = TRUE,
                           palette     = unname(group_colors),
                           risk.table  = TRUE, risk.table.cex = 1.1,
                           title       = paste0("Supervivencia: ", G0, " vs ", G1, " — ", stype),
                           xlab        = "Tiempo (días)", ylab = "Probabilidad de Supervivencia",
                           legend.labs = GROUP_NAMES,
                           theme       = base_theme +
                             theme(plot.title = element_text(size = 22, face = "bold")))),
          error = function(e) warning(paste("KM subtipo", stype, "falló:", e$message)))
      }
    }
  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier.")
  }
} else {
  warning("⚠️  Columnas OS y/o OS.time no encontradas.")
}

dev.off()

cat("\n✅ Reporte generado:", pdf_name, "\n")
cat("✅ Tabla estadística: 'Estadisticas_Metabolicas_Clusters.csv'\n")
cat("✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'\n")

🔬 Datos metabólicos cargados: 1226 filas, 447 columnas
📋 Datos clínicos cargados: 1225 filas, 121 columnas

🔗 Modelos en común: 1225 
   Solo en metabólico: 1 
   Solo en clínico: 0 

✅ Dataset merged: 1226 filas, 567 columnas

🔍 Valores únicos en columna de cluster (raw):

  -1    0    1 <NA> 
   8 1107  111    0 

✅ Pacientes por cluster (ruido DBSCAN -1 excluido):

Cluster 0 Cluster 1 
     1107       111 

🔍 Columnas duplicadas (.clin) eliminadas: 15 

🔍 Subsistemas SA detectados: 201 
   Subsistemas únicos: 85 
   • SA_Alanine_and_aspartate_metabolism
   • SA_Aminosugar_metabolism
   • SA_Androgen_and_estrogen_synthesis_and_metabolism
   • SA_Arachidonic_acid_metabolism
   • SA_Arginine_and_proline_metabolism
   • SA_Beta_Alanine_metabolism
   • SA_Bile_acid_synthesis
   • SA_Biomass_reactions
   • SA_Biotin_metabolism
   • SA_Butanoate_metabolism 
   ... 75 más

📊 Variables metabólicas encontradas: 248 / 252 
⚠️  NO encontradas ( 4 ):
   • LipidSat_pFBA
   • LipidUnsat_pFBA
   • 

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → TNBC → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → TNBC → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → TNBC → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → TNBC → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Tamaño de efecto: Cliff's δ  |  ≥0.147 pequeño 


📊 KM pacientes: 1195 ( Cluster 0 : 1086 |  Cluster 1 : 109 )


Ignoring unknown labels:
• colour : "Strata"
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Cluster 0 vs Cluster 1 — Luminal_HER2+' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Cluster 0 vs Cluster 1 — Luminal_HR+' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Cluster 0 vs Cluster 1 — Triple_Negative' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Cluster 0 vs Cluster 1 — HER2_Enriched' in 'mbcsToSbcs': - substituted for — (U+2014)”


pdf 
  2


✅ Reporte generado: Reporte_UMAP_N50_D05_C2_Meuclidean_S100_DBSCAN_K2_S0_98_DB0_02_CH423206_Seed42.pdf 
✅ Tabla estadística: 'Estadisticas_Metabolicas_Clusters.csv'
✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'


In [ ]:
# Pega el output de esto:
cat("\n🔍 Valores únicos en ER_PR_HER2_Combo:\n")
print(table(analysis$ER_PR_HER2_Combo, useNA = "always"))


🔍 Valores únicos en ER_PR_HER2_Combo:

  HER2_Enriched   Luminal_HER2+     Luminal_HR+ Triple_Negative            <NA> 
             73             280             576             237               0 
